# GreenAudioBench — official Colab T4 runner

Thin stage runner. **Runtime → Change runtime type → T4 GPU** first.

Run cells top to bottom. Set `STAGE` in the config cell:

| STAGE | What runs | Needs |
|---|---|---|
| `smoke` | tiny CPU/GPU smoke of every stage, writes only to `data/smoke/` | — |
| `m2` | ESC-50 embedding extraction (all 5 models, fp32) | ESC-50 (auto-downloaded) |
| `m3` | linear probes on cached embeddings | `m2` caches |
| `m4` | zero-shot CLAP (both models, 2 templates) | ESC-50 (+`m2` cache speeds it up) |
| `m5` | latency + NVML energy (fp32+fp16, batch 1/32) | ESC-50 |

Stages are idempotent; caches and results survive re-runs. With
`USE_DRIVE=True` embedding caches + results also persist across Colab VMs.

In [ ]:
# ------------------------- CONFIG -------------------------
REPO_URL = ""  # REQUIRED: https URL of the GreenAudioBench git repository
BRANCH = "main"
STAGE = "smoke"          # smoke | m2 | m3 | m4 | m5
USE_DRIVE = False         # persist data/embeddings + results/ to Google Drive
ALLOW_DIRTY = False       # debug override for the clean-tree guard (avoid)
assert REPO_URL, "Set REPO_URL to the repository https URL before running"

In [ ]:
# ------------------- GPU / driver check -------------------
!nvidia-smi
import torch
assert torch.cuda.is_available(), "No CUDA device — select the T4 GPU runtime"
name = torch.cuda.get_device_name(0)
print("GPU:", name, "| torch", torch.__version__, "| CUDA", torch.version.cuda)
if "T4" not in name:
    print("WARNING: official numbers are defined on Tesla T4 — this GPU is", name)

In [ ]:
# ---------------- clone/pull + dependencies ----------------
import os
if not os.path.isdir("greenaudiobench"):
    !git clone --branch {BRANCH} {REPO_URL} greenaudiobench
%cd greenaudiobench
!git pull --ff-only
!pip install -q -r env/requirements.txt
!pip freeze > env/requirements-lock-colab.txt
!git log --oneline -1 && git status --porcelain | head -5
import subprocess
dirty = subprocess.run(["git", "status", "--porcelain"], capture_output=True, text=True).stdout.strip()
print("working tree:", "DIRTY (official runs will refuse)" if dirty else "clean")

In [ ]:
# ------- optional: persist caches/results to Drive ---------
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    import os
    persist = "/content/drive/MyDrive/greenaudiobench"
    for sub in ("data/embeddings", "results", "figures"):
        src = os.path.join(persist, sub)
        os.makedirs(src, exist_ok=True)
        if os.path.isdir(sub) and not os.path.islink(sub):
            !cp -rn {sub}/. {src}/ 2>/dev/null || true
            !rm -rf {sub}
        if not os.path.islink(sub):
            os.makedirs(os.path.dirname(sub), exist_ok=True)
            os.symlink(src, sub)
    print("persisted dirs linked to", persist)

In [ ]:
# ----------------- data (ESC-50 only, M1) ------------------
if STAGE in ("smoke", "m2", "m4", "m5"):
    !python scripts/download_data.py --datasets esc50

In [ ]:
# ---------------------- run the stage ----------------------
dirty_flag = "--allow-dirty" if ALLOW_DIRTY else ""
cmds = {
    "smoke": [
        "python -m pytest -q",
        "python scripts/extract_embeddings.py --dataset esc50 --smoke 8",
        "python scripts/run_probes.py --dataset esc50 --smoke",
        f"python scripts/run_zeroshot.py --dataset esc50 --smoke 8",
        f"python scripts/measure_efficiency.py --smoke",
    ],
    "m2": [f"python scripts/extract_embeddings.py --dataset esc50 --batch-size 16 {dirty_flag}"],
    "m3": [f"python scripts/run_probes.py --dataset esc50 {dirty_flag}"],
    "m4": [f"python scripts/run_zeroshot.py --dataset esc50 {dirty_flag}"],
    "m5": [f"python scripts/measure_efficiency.py {dirty_flag}"],
}
for cmd in cmds[STAGE]:
    print("\n>>>", cmd, flush=True)
    rc = os.system(cmd)
    assert rc == 0, f"stage command failed: {cmd}"

In [ ]:
# ------------- collect artifacts for download --------------
!mkdir -p /content/artifacts
!cp -r results /content/artifacts/ 2>/dev/null || true
!cp env/requirements-lock-colab.txt /content/artifacts/ 2>/dev/null || true
!ls -laR data/embeddings 2>/dev/null | head -30
!cd /content && zip -qr artifacts.zip artifacts && ls -la artifacts.zip
print("Download /content/artifacts.zip via the Files panel, or use Drive persistence.")